In [1]:
from datasets import Dataset, DatasetDict
from pathlib import Path
import numpy as np
import os

e:\LLM_Fintune\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from glob import glob

def uniform_downsample(motion, target_frames=40):
    """
    motion: shape (T, 21, 3)
    return: shape (target_frames, 21, 3)
    """
    T = motion.shape[0]
    idx = np.linspace(0, T - 1, target_frames).astype(int)
    return motion[idx]

def downsample_all_npy_files(data_dir, output_data_dir, target_frames=40):
    os.makedirs(output_data_dir, exist_ok=True)
    npy_files = sorted(glob(os.path.join(data_dir, "*.npy")))
    for f in npy_files:
        data = np.load(f) 
        if data.shape[0] > target_frames:
            resized = uniform_downsample(data, target_frames=target_frames)
        else:
            resized = data
        out_path = os.path.join(output_data_dir, os.path.basename(f))
        np.save(out_path, resized)
        # print(resized.shape)
    print("complete")

In [3]:
data_dir = Path("KIT-ML/new_joints")  
output_data_dir = Path("KIT-ML/new_joints_40")
downsample_all_npy_files(data_dir, output_data_dir, target_frames=40)

complete


In [4]:
def quantize_motion(continuous_motion, bins=64):
    continuous_motion = continuous_motion.astype(float) 
    v_min = continuous_motion.min()
    v_max = continuous_motion.max() 

    eps = 1e-8
    v_range = np.maximum(v_max - v_min, eps)
    norm = (continuous_motion - v_min) / v_range

    quantized_motion = np.round(norm * (bins - 1)).astype(np.int32)
    quantized_motion = np.clip(quantized_motion, 0, bins - 1)

    return quantized_motion

def quantize_all_motion(data_dir, output_data_dir, bins=64):
    os.makedirs(output_data_dir, exist_ok=True)
    npy_files = sorted(glob(os.path.join(data_dir, "*.npy")))
    for f in npy_files:
        data = np.load(f)
        data_quantized = quantize_motion(data, bins=bins)
        out_path = os.path.join(output_data_dir, os.path.basename(f))
        np.save(out_path, data_quantized)
    print("complete")


In [5]:
data_dir = Path("KIT-ML/new_joints_40")  
output_data_dir = Path("KIT-ML/new_joints_40_bin256")
quantize_all_motion(data_dir, output_data_dir, bins=256)

complete


In [6]:
num_40 = len(list(data_dir.glob("*.npy")))
print("Number of .npy files:", num_40)

num_40_bin256 = len(list(output_data_dir.glob("*.npy")))
print("Number of .npy files:", num_40_bin256)

Number of .npy files: 7812
Number of .npy files: 7812


In [7]:
def load_split(data_dir: Path, split_name: str) -> Dataset:
    split_file = data_dir / f"{split_name}.txt"
    id_list = split_file.read_text().strip().splitlines()
    # num_ids_used = len(id_list) * 0.01
    # id_list = id_list[:int(num_ids_used)]
    records = []
    for motion_id in id_list:
        try: # not all filenames listed in train.txt have a corresponding npy file
            motion_orig = np.load(data_dir / "new_joints" / f"{motion_id}.npy") # (T, 21, 3)
        except FileNotFoundError:
            print(f"new_joints/{motion_id}.npy not found")
            continue
        if motion_orig.shape[0] < 120:
            motion = np.load(data_dir / "new_joints_40_bin256" / f"{motion_id}.npy")  # (<=40, 21, 3)

            with open(data_dir / "texts" / f"{motion_id}.txt", "r") as f:
                first_line = f.readline().strip()
            text = first_line.split("#")[0]

            records.append(
                {
                    "motion_id": motion_id,
                    "text": text,
                    "motion": motion,
                }
            )

    return Dataset.from_list(records)

data_dir = Path("kit-ml")
ds = DatasetDict(
    train=load_split(data_dir, "train"),
    validation=load_split(data_dir, "val"),
    test=load_split(data_dir, "test"),
)

new_joints/00942.npy not found
new_joints/M00942.npy not found


In [9]:
ds

DatasetDict({
    train: Dataset({
        features: ['motion_id', 'text', 'motion'],
        num_rows: 4130
    })
    validation: Dataset({
        features: ['motion_id', 'text', 'motion'],
        num_rows: 258
    })
    test: Dataset({
        features: ['motion_id', 'text', 'motion'],
        num_rows: 686
    })
})

In [10]:
example = ds["train"][18]
example["motion_id"], example["text"], len(example["motion"]), type(example["motion"])

('03202',
 'A person walks straight forwards, turns around and then walks back.',
 40,
 list)

In [12]:
import json

def format_for_qwen(example):
    # 1. 系统提示
    system_msg = {"role": "system", "content": "You are a motion generation assistant. Generate the motion sequence (flattened integer tokens) based on the text description."}
    
    # 2. 用户输入 (文本)
    user_msg = {"role": "user", "content": example["text"]}
    
    # 3. 助手输出 (动作)
    # 将 (T, 21, 3) 的数组展平，并转换为用空格分隔的字符串
    motion_flat = np.array(example["motion"]).flatten()
    motion_str = " ".join(map(str, motion_flat))
    assistant_msg = {"role": "assistant", "content": motion_str}
    
    return {
        "messages": [system_msg, user_msg, assistant_msg]
    }

# 转换数据集
# 注意：这会处理 train, validation, test 所有部分
qwen_ds = ds.map(format_for_qwen, remove_columns=ds["train"].column_names)

# 保存为 JSONL 文件
output_dir = Path("KIT-ML/qwen_ready")
output_dir.mkdir(exist_ok=True)

qwen_ds["train"].to_json(output_dir / "train.jsonl", orient="records", lines=True, force_ascii=False)
qwen_ds["validation"].to_json(output_dir / "val.jsonl", orient="records", lines=True, force_ascii=False)
qwen_ds["test"].to_json(output_dir / "test.jsonl", orient="records", lines=True, force_ascii=False)

print(f"Data saved to {output_dir}")
print("Example entry:")
print(json.dumps(qwen_ds["train"][0], indent=2))

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 29.41ba/s]

Data saved to KIT-ML\qwen_ready
Example entry:
{
  "messages": [
    {
      "content": "You are a motion generation assistant. Generate the motion sequence (flattened integer tokens) based on the text description.",
      "role": "system"
    },
    {
      "content": "A person who is laughing hard.",
      "role": "user"
    },
    {
      "content": "60 157 60 59 171 63 60 192 65 60 200 64 60 204 67 41 203 63 32 181 40 30 156 33 79 197 66 82 173 44 76 150 34 51 156 62 37 116 66 26 75 57 25 71 61 22 68 70 69 159 59 83 118 54 88 77 42 91 73 45 97 69 51 60 157 60 59 171 63 61 192 65 60 200 64 61 204 67 41 203 63 31 182 40 28 158 33 80 197 66 82 173 43 75 151 32 51 156 62 35 117 67 23 76 57 21 73 61 18 69 69 69 159 59 83 118 54 87 77 43 90 73 45 96 69 52 60 157 61 59 171 63 61 192 65 61 200 64 62 204 67 42 203 63 31 183 40 27 158 33 80 196 65 82 173 42 75 151 32 51 156 62 33 117 67 20 77 57 18 74 60 15 70 69 69 159 59 83 118 54 87 77 43 90 73 45 96 69 52 60 157 61 59 171 63 62 192 65 61